# Article 2 - Early Forecasting of Irrigation Demand from Multivariate IoT Greenhouse Time Series under Strict Chronological Evaluation

**Reproduction notebook.** 10/30/60-min irrigation early warning with causal features and a strict train / validation / test split in time (Tables I, III-VII, Figs. 1-3).

**Run time on Colab:** about 5 min.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badrelkhalylyphd-boop/irrigation-early-warning-forecasting/blob/main/reproduce_colab.ipynb)

### How to use
1. **Runtime > Run all.** The first cell installs the exact library versions used for the paper. On Colab the session then restarts by itself.
2. Upload `greenhouse_dataset.csv` in the **Files** panel of Colab (folder icon on the left; it goes to `/content`). The file is in `data/` of this repository or on [Mendeley Data](https://doi.org/10.17632/2tvpsht4s4.1). It survives the automatic restart of step 1.
3. **Runtime > Run all** a second time. Cell 2 reads `/content/greenhouse_dataset.csv` directly and checks its SHA-256.
4. The last section compares every number of the revised paper with this run and marks each one ✅ or ❌.

Locally (Jupyter), open the notebook from the repository root: the data file is picked up from `data/`.

## 1. Environment

In [ ]:
# Install the exact library versions used for the paper (runs once).
import os, sys, subprocess, importlib.metadata as md
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
PINNED = {'numpy': '2.2.6', 'pandas': '2.3.3', 'scikit-learn': '1.7.2', 'scipy': '1.15.3', 'lightgbm': '4.7.0', 'matplotlib': '3.11.2', 'xgboost': '3.2.0', 'catboost': '1.2.10'}

def _version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

wrong = {p: _version(p) for p, v in PINNED.items() if _version(p) != v}
if wrong:
    print("Installing pinned versions (currently:", wrong, ") ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + [f"{p}=={v}" for p, v in PINNED.items()])
    if IN_COLAB:
        print("\nDone. The session restarts now. Then choose Runtime > Run all again.")
        os.kill(os.getpid(), 9)
    raise SystemExit("Pinned versions installed. Restart the kernel, then run all cells again.")
print("Environment OK -", sys.version.split()[0], PINNED)


## 2. Data

In [ ]:
# Get the dataset.
# On Colab: upload greenhouse_dataset.csv in the Files panel (left sidebar); it lands in /content and is read from there.
# Locally: the copy in data/ of the repository is used.
import hashlib, shutil, urllib.request
from pathlib import Path

COLAB_FILE = Path("/content/greenhouse_dataset.csv")   # file uploaded through the Colab Files panel
DATA_URL = ""  # optional, e.g. "https://raw.githubusercontent.com/badrelkhalylyphd-boop/irrigation-early-warning-forecasting/main/data/greenhouse_dataset.csv"
EXPECTED_SHA256 = "e64a904a981d323f67210138cfc078e98d1911e6d2bb42195bf76aa818c51aff"

WORK = Path("work").resolve()
SRC, OUT = WORK / "src", WORK / "outputs"
SRC.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
DATA = WORK / "greenhouse_dataset.csv"

candidates = [COLAB_FILE, *sorted(Path("/content").glob("greenhouse_dataset*.csv")),
              Path("data/greenhouse_dataset.csv"), Path("../data/greenhouse_dataset.csv"), Path("greenhouse_dataset.csv")]
found = [p for p in candidates if p.is_file()]
if found:
    shutil.copy(found[0], DATA)
    print("Dataset read from", found[0])
elif DATA.exists():
    print("Dataset already in", DATA)
elif DATA_URL:
    urllib.request.urlretrieve(DATA_URL, DATA)
elif IN_COLAB:
    from google.colab import files
    print("greenhouse_dataset.csv not found in /content. Select it now (or upload it in the Files panel and rerun this cell).")
    name = next(iter(files.upload()))
    Path(name).replace(DATA)
else:
    raise FileNotFoundError("Put greenhouse_dataset.csv in data/ or next to this notebook, or set DATA_URL.")

sha = hashlib.sha256(DATA.read_bytes()).hexdigest()
print("SHA-256:", sha)
print("Same file as in the paper." if sha == EXPECTED_SHA256 else "WARNING: this is not the file used in the paper.")


## 3. Code
The cells below write the scripts of `src/` unchanged into `work/src/`.

In [ ]:
%%writefile work/src/prep2.py
"""Article 2 - causal one-minute feature construction and future-irrigation targets."""
import argparse
import numpy as np
import pandas as pd
SENSORS = ['Soil Percent','Soil Raw','Suhu Tanah','Suhu Udara','Humidity']

def parse_timestamp(series):
    s = series.astype(str)
    s = s.str.replace(
        r'^(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})$',
        lambda m: m.group(1) + ' ' + m.group(2).replace('-', ':'),
        regex=True,
    )
    return pd.to_datetime(s, errors='coerce')

def future_target(frame, horizon_min):
    target = np.zeros(len(frame), dtype=np.int8)
    for _, idx in frame.groupby('segment').groups.items():
        idx = np.asarray(list(idx))
        t = frame.loc[idx, 'ts'].values.astype('datetime64[ns]')
        y = frame.loc[idx, 'Label'].values
        events = t[y == 1]
        if len(events) == 0:
            continue
        pos = np.searchsorted(events, t, side='right')
        valid = pos < len(events)
        next_event = np.empty(len(t), dtype='datetime64[ns]')
        next_event[:] = np.datetime64('NaT')
        next_event[valid] = events[pos[valid]]
        delta = (next_event - t).astype('timedelta64[s]').astype(float)
        target[idx] = (valid & (delta > 0) & (delta <= horizon_min * 60)).astype(np.int8)
    return target

def main(inp, out):
    d = pd.read_csv(inp, sep=';')
    d['ts'] = parse_timestamp(d['Timestamp'])
    for c in SENSORS + ['Label']:
        d[c] = pd.to_numeric(d[c], errors='coerce')
    d = d[d['Label'].notna() & d['ts'].notna()].copy()
    d = d.dropna(subset=SENSORS).sort_values('ts')
    # One-minute grid. Sensor values are means; any irrigation reading marks the minute.
    agg = {c: 'mean' for c in SENSORS}
    agg['Label'] = 'max'
    m = d.set_index('ts').groupby(pd.Grouper(freq='1min')).agg(agg)
    m = m.dropna(subset=SENSORS + ['Label']).reset_index()
    # Do not let rolling windows or future targets cross collection gaps.
    m['gap_min'] = m['ts'].diff().dt.total_seconds().div(60)
    m['segment'] = (m['gap_min'].isna() | (m['gap_min'] > 2.1)).cumsum()
    for w in (5, 15):
        g = m.groupby('segment', group_keys=False)
        for c in SENSORS:
            minp = max(2, w // 2)
            m[f'{c}_mean{w}'] = g[c].transform(lambda s: s.rolling(w, min_periods=minp).mean())
            m[f'{c}_std{w}'] = g[c].transform(lambda s: s.rolling(w, min_periods=minp).std())
    for c in SENSORS:
        m[f'{c}_diff1'] = m.groupby('segment')[c].diff(1)
        m[f'{c}_diff5'] = m.groupby('segment')[c].diff(5)
    minute_of_day = m['ts'].dt.hour * 60 + m['ts'].dt.minute
    m['sin_time'] = np.sin(2*np.pi*minute_of_day/1440)
    m['cos_time'] = np.cos(2*np.pi*minute_of_day/1440)
    for H in (10, 30, 60):
        m[f'y{H}'] = future_target(m, H)
    # Early-warning samples: the irrigation has not started at the current minute.
    m = m[m['Label'] == 0].copy()
    non_features = {'ts','Label','gap_min','segment','y10','y30','y60'}
    features = [c for c in m.columns if c not in non_features]
    m = m.dropna(subset=features).reset_index(drop=True)
    m.to_csv(out, index=False)
    print('eligible samples:', len(m), 'segments:', m['segment'].nunique(), 'features:', len(features))
    for H in (10,30,60):
        print('H=',H,'positives=',int(m[f'y{H}'].sum()))
        masks = {
            'train': m['ts'] < '2026-05-19',
            'validation': (m['ts'] >= '2026-05-19') & (m['ts'] < '2026-05-20'),
            'test': m['ts'] >= '2026-05-20',
        }
        for name, mask in masks.items():
            z=m[mask]
            print(name,len(z),int(z[f'y{H}'].sum()))

if __name__ == '__main__':
    ap=argparse.ArgumentParser(); ap.add_argument('input_csv'); ap.add_argument('--out',default='forecast_features.csv')
    a=ap.parse_args(); main(a.input_csv,a.out)


In [ ]:
%%writefile work/src/eval2.py
"""Article 2 - chronological training/validation/test evaluation for 10/30/60 min horizons."""
import argparse, json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, average_precision_score, roc_auc_score, confusion_matrix,
    precision_recall_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
EXCLUDE={'ts','Label','gap_min','segment','y10','y30','y60'}
CURRENT=['Soil Percent','Soil Raw','Suhu Tanah','Suhu Udara','Humidity']
def choose_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f = 2*pr*rc/np.maximum(pr+rc,1e-12)
    i = int(np.nanargmax(f[:-1]))
    return float(th[i]), float(f[i])
def score(y,p,t):
    z=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,z,labels=[0,1]).ravel()
    return dict(accuracy=accuracy_score(y,z),balanced_accuracy=balanced_accuracy_score(y,z),
        precision=precision_score(y,z,zero_division=0),recall=recall_score(y,z,zero_division=0),
        f1=f1_score(y,z,zero_division=0),mcc=matthews_corrcoef(y,z),
        pr_auc=average_precision_score(y,p),roc_auc=roc_auc_score(y,p),
        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp),threshold=float(t))
def main(inp,out):
    d=pd.read_csv(inp,parse_dates=['ts']); F=[c for c in d.columns if c not in EXCLUDE]
    tr=d[d.ts<'2026-05-19']; va=d[(d.ts>='2026-05-19')&(d.ts<'2026-05-20')]; te=d[d.ts>='2026-05-20']
    R={'meta':{'n_train':len(tr),'n_validation':len(va),'n_test':len(te),'n_features':len(F)},'horizons':{},'ablation':{}}
    for H in (10,30,60):
        ytr=tr[f'y{H}'].to_numpy(); yv=va[f'y{H}'].to_numpy(); yt=te[f'y{H}'].to_numpy()
        ratio=(len(ytr)-ytr.sum())/max(1,ytr.sum())
        models={
            'Logistic':make_pipeline(StandardScaler(),LogisticRegression(max_iter=5000,class_weight='balanced',random_state=42)),
            'Random Forest':RandomForestClassifier(n_estimators=300,min_samples_leaf=2,class_weight='balanced_subsample',n_jobs=-1,random_state=42),
            'Extra Trees':ExtraTreesClassifier(n_estimators=300,min_samples_leaf=2,class_weight='balanced',n_jobs=-1,random_state=42),
            'LightGBM':lgb.LGBMClassifier(n_estimators=250,learning_rate=.03,num_leaves=15,scale_pos_weight=ratio,random_state=42,verbosity=-1),
            'XGBoost':xgb.XGBClassifier(n_estimators=250,learning_rate=.03,max_depth=4,subsample=.9,colsample_bytree=.9,scale_pos_weight=ratio,eval_metric='logloss',random_state=42,n_jobs=-1),
            'CatBoost':CatBoostClassifier(iterations=250,learning_rate=.03,depth=6,auto_class_weights='Balanced',verbose=False,random_seed=42,thread_count=-1),
        }
        R['horizons'][str(H)]={}
        for name,m in models.items():
            t0=time.time(); m.fit(tr[F],ytr); fit=time.time()-t0
            pv=m.predict_proba(va[F])[:,1]; thr,val_f1=choose_threshold(yv,pv)
            pt=m.predict_proba(te[F])[:,1]; met=score(yt,pt,thr); met['validation_f1']=val_f1; met['fit_s']=fit
            R['horizons'][str(H)][name]=met
            print(H,name,'F1',round(met['f1'],4),'MCC',round(met['mcc'],4),'PR-AUC',round(met['pr_auc'],4))
        # Ablation: same logistic estimator, current readings only versus all causal temporal features.
        R['ablation'][str(H)]={}
        for name,cols in [('Current-only',CURRENT),('Current+temporal',F)]:
            m=make_pipeline(StandardScaler(),LogisticRegression(max_iter=5000,class_weight='balanced',random_state=42))
            m.fit(tr[cols],ytr); pv=m.predict_proba(va[cols])[:,1]; thr,vf=choose_threshold(yv,pv); pt=m.predict_proba(te[cols])[:,1]
            met=score(yt,pt,thr); met['validation_f1']=vf; R['ablation'][str(H)][name]=met
    json.dump(R,open(out,'w'),indent=2)
if __name__=='__main__':
    ap=argparse.ArgumentParser(); ap.add_argument('features_csv'); ap.add_argument('--out',default='forecast_results.json')
    a=ap.parse_args(); main(a.features_csv,a.out)


In [ ]:
%%writefile work/src/bootstrap2.py
"""Article 2 - 2,000-replicate bootstrap intervals for fixed logistic-regression test predictions."""
import argparse, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (precision_recall_curve, f1_score, matthews_corrcoef,
    average_precision_score, balanced_accuracy_score, precision_score, recall_score)
EXCLUDE={'ts','Label','gap_min','segment','y10','y30','y60'}
def choose_threshold(y,p):
    pr,rc,th=precision_recall_curve(y,p)
    f=2*pr*rc/np.maximum(pr+rc,1e-12)
    i=int(np.nanargmax(f[:-1]))
    return float(th[i])
def main(inp,out,nboot=2000,seed=20261016):
    d=pd.read_csv(inp,parse_dates=['ts']); F=[c for c in d.columns if c not in EXCLUDE]
    tr=d[d.ts<'2026-05-19']; va=d[(d.ts>='2026-05-19')&(d.ts<'2026-05-20')]; te=d[d.ts>='2026-05-20']
    rng=np.random.default_rng(seed); R={}
    for H in (10,30,60):
        ytr=tr[f'y{H}'].to_numpy(); yv=va[f'y{H}'].to_numpy(); yt=te[f'y{H}'].to_numpy()
        mdl=make_pipeline(StandardScaler(),LogisticRegression(max_iter=5000,class_weight='balanced',random_state=42))
        mdl.fit(tr[F],ytr)
        thr=choose_threshold(yv,mdl.predict_proba(va[F])[:,1])
        p=mdl.predict_proba(te[F])[:,1]; z=(p>=thr).astype(int)
        vals={k:[] for k in ('f1','mcc','prauc','bacc','precision','recall')}
        for _ in range(nboot):
            idx=rng.integers(0,len(yt),len(yt)); yy=yt[idx]; pp=p[idx]; zz=z[idx]
            if len(np.unique(yy))<2: continue
            vals['f1'].append(f1_score(yy,zz,zero_division=0))
            vals['mcc'].append(matthews_corrcoef(yy,zz))
            vals['prauc'].append(average_precision_score(yy,pp))
            vals['bacc'].append(balanced_accuracy_score(yy,zz))
            vals['precision'].append(precision_score(yy,zz,zero_division=0))
            vals['recall'].append(recall_score(yy,zz,zero_division=0))
        R[str(H)]={'threshold':thr,'ci95':{k:[float(np.quantile(v,.025)),float(np.quantile(v,.975))] for k,v in vals.items()}}
    json.dump(R,open(out,'w'),indent=2)
if __name__=='__main__':
    ap=argparse.ArgumentParser(); ap.add_argument('features_csv'); ap.add_argument('--out',default='bootstrap_results.json')
    ap.add_argument('--nboot',type=int,default=2000)
    a=ap.parse_args(); main(a.features_csv,a.out,a.nboot)


In [ ]:
%%writefile work/src/figs2.py
"""Article 2 - generate paper figures from the saved JSON and feature table."""
import argparse, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def main(features,results,outdir):
    import os; os.makedirs(outdir,exist_ok=True)
    d=pd.read_csv(features,parse_dates=['ts']); R=json.load(open(results))
    # Fig 1: positives by split and horizon
    labels=['Train','Validation','Test']; x=np.arange(3); w=.24
    fig,ax=plt.subplots(figsize=(5.4,3.1))
    masks=[d.ts<'2026-05-19',(d.ts>='2026-05-19')&(d.ts<'2026-05-20'),d.ts>='2026-05-20']
    for j,H in enumerate((10,30,60)):
        vals=[int(d.loc[m,f'y{H}'].sum()) for m in masks]; ax.bar(x+(j-1)*w,vals,width=w,label=f'{H} min')
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('Positive early-warning minutes'); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(outdir+'/fig1_target_counts.png',dpi=300); plt.close(fig)
    # Fig 2: model F1 by horizon
    models=['Logistic','Random Forest','Extra Trees','LightGBM','XGBoost','CatBoost']; x=np.arange(len(models)); w=.24
    fig,ax=plt.subplots(figsize=(6.3,3.2))
    for j,H in enumerate((10,30,60)):
        vals=[100*R['horizons'][str(H)][m]['f1'] for m in models]; ax.bar(x+(j-1)*w,vals,width=w,label=f'{H} min')
    ax.set_xticks(x); ax.set_xticklabels(models,rotation=25,ha='right'); ax.set_ylabel('Test F1-score (%)'); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(outdir+'/fig2_model_f1.png',dpi=300); plt.close(fig)
    # Fig 3: current-only vs temporal ablation
    fig,ax=plt.subplots(figsize=(5.2,3.0)); x=np.arange(3); w=.32
    for j,n in enumerate(['Current-only','Current+temporal']):
        vals=[100*R['ablation'][str(H)][n]['f1'] for H in (10,30,60)]; ax.bar(x+(j-.5)*w,vals,width=w,label=n)
    ax.set_xticks(x); ax.set_xticklabels(['10 min','30 min','60 min']); ax.set_ylabel('Test F1-score (%)'); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(outdir+'/fig3_temporal_ablation.png',dpi=300); plt.close(fig)

if __name__=='__main__':
    ap=argparse.ArgumentParser(); ap.add_argument('features_csv'); ap.add_argument('results_json'); ap.add_argument('--outdir',default='figures')
    a=ap.parse_args(); main(a.features_csv,a.results_json,a.outdir)


In [ ]:
# Helper: run a script from work/src inside work/outputs and stream its output.
def run(script, *args):
    cmd = [sys.executable, "-u", str(SRC / script), *map(str, args)]
    env = dict(os.environ, MPLBACKEND="Agg", PYTHONIOENCODING="utf-8")
    p = subprocess.Popen(cmd, cwd=OUT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"{script} failed")


## 4. Run the experiment
All outputs go to `work/outputs/`.

### Step 1 - one-minute grid, segments, 37 causal features, targets

In [ ]:
run("prep2.py", DATA, "--out", "forecast_features.csv")

### Step 2 - six models, validation thresholds, test scores, ablation

In [ ]:
run("eval2.py", "forecast_features.csv", "--out", "forecast_results.json")

### Step 3 - 2,000-replicate bootstrap

In [ ]:
run("bootstrap2.py", "forecast_features.csv", "--out", "bootstrap_results.json", "--nboot", 2000)

### Step 4 - figures

In [ ]:
run("figs2.py", "forecast_features.csv", "forecast_results.json", "--outdir", "figures")

## 5. Check against the revised paper

In [ ]:
# Compare every reported number with what this run produced.
import math
import pandas as pd
CHECKS = []

def chk(item, expected, obtained, tol=0.006):
    """Numbers are compared at the precision printed in the paper."""
    if isinstance(expected, str):
        ok = str(obtained) == expected
    else:
        ok = obtained is not None and not (isinstance(obtained, float) and math.isnan(obtained)) and abs(obtained - expected) <= tol
    CHECKS.append({"item": item, "paper": expected,
                   "this run": round(obtained, 4) if isinstance(obtained, float) else obtained,
                   "status": "✅" if ok else "❌"})

import json
F = pd.read_csv(OUT / "forecast_features.csv", parse_dates=["ts"])
R = json.load(open(OUT / "forecast_results.json")); B = json.load(open(OUT / "bootstrap_results.json"))
chk("Sec. III-D | eligible minutes", 7090, len(F), 0)
chk("Sec. III-D | segments", 30, F.segment.nunique(), 0)
masks = {"train": F.ts < "2026-05-19", "val": (F.ts >= "2026-05-19") & (F.ts < "2026-05-20"), "test": F.ts >= "2026-05-20"}
for part, n in (("train", 3587), ("val", 1312), ("test", 2191)):
    chk(f"Table I | {part} samples", n, int(masks[part].sum()), 0)
for H, pos in {10: (125, 84, 340), 30: (246, 204, 507), 60: (315, 336, 560)}.items():
    for part, p in zip(("train", "val", "test"), pos):
        chk(f"Table I | {H} min | {part} positives", p, int(F.loc[masks[part], f"y{H}"].sum()), 0)

LR = {"10": dict(f1=59.74, mcc=51.82, pr_auc=47.22), "30": dict(f1=63.42, mcc=55.40, pr_auc=64.06),
      "60": dict(f1=73.42, mcc=65.43, pr_auc=60.78, precision=79.01, recall=68.57, balanced_accuracy=81.16)}
for H, vals in LR.items():
    for k, v in vals.items():
        chk(f"Tables III/V | logistic {H} min | {k}", v, 100 * R["horizons"][H]["Logistic"][k])
for k, v in dict(tn=1529, fp=102, fn=176, tp=384).items():
    chk(f"Table V | logistic 60 min | {k}", v, R["horizons"]["60"]["Logistic"][k], 0)
for H, thr in {"10": 0.999980, "30": 1.000000, "60": 0.999991}.items():
    chk(f"Table IV | logistic {H} min | threshold", thr, R["horizons"][H]["Logistic"]["threshold"], 6e-7)

for model, vals in {"Random Forest": dict(f1=52.68, recall=88.57, precision=37.49), "Extra Trees": dict(f1=42.45),
                    "LightGBM": dict(f1=43.68), "XGBoost": dict(f1=45.92),
                    "CatBoost": dict(f1=51.54, recall=94.11, precision=35.49)}.items():
    for k, v in vals.items():
        chk(f"Fig. 1 / Sec. IV-A | {model} 60 min | {k}", v, 100 * R["horizons"]["60"][model][k])

for H, f1 in {"10": 26.87, "30": 42.57, "60": 40.90}.items():
    chk(f"Table VII | current-only {H} min | F1", f1, 100 * R["ablation"][H]["Current-only"]["f1"])
chk("Sec. IV-E | current-only 60 min | MCC", 7.60, 100 * R["ablation"]["60"]["Current-only"]["mcc"])
chk("Sec. IV-E | current-only 60 min | PR-AUC", 22.48, 100 * R["ablation"]["60"]["Current-only"]["pr_auc"])

CI = {"10": dict(f1=(55.42, 63.65), mcc=(47.18, 56.30)), "30": dict(f1=(59.74, 66.89), mcc=(51.19, 59.44)),
      "60": dict(f1=(70.26, 76.40), mcc=(61.51, 69.13), prauc=(56.18, 65.32))}
for H, d in CI.items():
    for k, (lo, hi) in d.items():
        chk(f"Table VI | {H} min | {k} 95% CI low", lo, 100 * B[H]["ci95"][k][0])
        chk(f"Table VI | {H} min | {k} 95% CI high", hi, 100 * B[H]["ci95"][k][1])


In [ ]:
report = pd.DataFrame(CHECKS)
n_ok = (report.status == "✅").sum()
print(f"{n_ok} / {len(report)} values match the revised paper.")
pd.set_option("display.max_rows", 500)
report[report.status == "❌"] if n_ok < len(report) else print("All checks passed.")


Full list of checks:

In [ ]:
report

## 6. Figures

In [ ]:
# Figures produced by this run
from IPython.display import Image, display
for f in sorted(OUT.rglob("*.png")):
    print(f.name); display(Image(filename=str(f), width=520))
